1. Get the data from collectapi -- gaspricesapi -- get/stateUsaPrice

In [15]:
from dotenv import load_dotenv
import os
import json
import pandas as pd
from sqlalchemy import create_engine , text

In [2]:
import http.client

load_dotenv()

conn = http.client.HTTPSConnection("api.collectapi.com")

api_token = os.getenv("API_TOKEN")

headers = {
    'content-type': "application/json",
    'authorization': f"apikey {api_token}"
    }

conn.request("GET", "/gasPrice/stateUsaPrice?state=WA", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8")) 

{"success":true,"result":{"state":[{"currency":"usd","name":"Washington","lowername":"washington","gasoline":"5.2140","midGrade":"5.5270","premium":"5.7740","diesel":"6.0940"}],"cities":[{"currency":"usd","gasoline":"4.9370","midGrade":"5.1930","premium":"5.4940","diesel":"5.6910","name":"Bellingham","lowername":"bellingham"},{"currency":"usd","gasoline":"5.4390","midGrade":"5.7580","premium":"5.9610","diesel":"6.1710","name":"Bremerton","lowername":"bremerton"},{"currency":"usd","gasoline":"5.1950","midGrade":"5.4150","premium":"5.6270","diesel":"6.2380","name":"Olympia","lowername":"olympia"},{"currency":"usd","gasoline":"4.9390","midGrade":"5.2600","premium":"5.5090","diesel":"5.8050","name":"Richland-Kennewick-Pasco","lowername":"richland-kennewick-pasco"},{"currency":"usd","gasoline":"5.5320","midGrade":"5.8170","premium":"6.0480","diesel":"6.3820","name":"Seattle-Bellevue-Everett","lowername":"seattle-bellevue-everett"},{"currency":"usd","gasoline":"4.6890","midGrade":"4.9580","p

2. extract city level data, drop the lowername column, rename name to cities  then you store to db

In [3]:
print(type(data))
print(res.status)
print(data[:200])

<class 'bytes'>
200
b'{"success":true,"result":{"state":[{"currency":"usd","name":"Washington","lowername":"washington","gasoline":"5.2140","midGrade":"5.5270","premium":"5.7740","diesel":"6.0940"}],"cities":[{"currency":"'


In [4]:
data =json.loads(data.decode("utf-8"))

print(type(data))

<class 'dict'>


In [ ]:
cities = data['result']['cities']
print(cities)

[{'currency': 'usd', 'gasoline': '4.9370', 'midGrade': '5.1930', 'premium': '5.4940', 'diesel': '5.6910', 'name': 'Bellingham', 'lowername': 'bellingham'}, {'currency': 'usd', 'gasoline': '5.4390', 'midGrade': '5.7580', 'premium': '5.9610', 'diesel': '6.1710', 'name': 'Bremerton', 'lowername': 'bremerton'}, {'currency': 'usd', 'gasoline': '5.1950', 'midGrade': '5.4150', 'premium': '5.6270', 'diesel': '6.2380', 'name': 'Olympia', 'lowername': 'olympia'}, {'currency': 'usd', 'gasoline': '4.9390', 'midGrade': '5.2600', 'premium': '5.5090', 'diesel': '5.8050', 'name': 'Richland-Kennewick-Pasco', 'lowername': 'richland-kennewick-pasco'}, {'currency': 'usd', 'gasoline': '5.5320', 'midGrade': '5.8170', 'premium': '6.0480', 'diesel': '6.3820', 'name': 'Seattle-Bellevue-Everett', 'lowername': 'seattle-bellevue-everett'}, {'currency': 'usd', 'gasoline': '4.6890', 'midGrade': '4.9580', 'premium': '5.1600', 'diesel': '5.7540', 'name': 'Spokane', 'lowername': 'spokane'}, {'currency': 'usd', 'gasoli

In [11]:
cities = pd.DataFrame(cities)

In [12]:
cities.head()

,currency,gasoline,midGrade,premium,diesel,name,lowername
0,usd,4.9370,5.1930,5.4940,5.6910,Bellingham,bellingham
1,usd,5.4390,5.7580,5.9610,6.1710,Bremerton,bremerton
2,usd,5.1950,5.4150,5.6270,6.2380,Olympia,olympia
3,usd,4.9390,5.2600,5.5090,5.8050,Richland-Kennewick-Pasco,richland-kennewick-pasco
4,usd,5.5320,5.8170,6.0480,6.3820,Seattle-Bellevue-Everett,seattle-bellevue-everett


In [13]:
cities = cities.rename(columns = {'lowername':'cities'})
cities.head()

,currency,gasoline,midGrade,premium,diesel,name,cities
0,usd,4.9370,5.1930,5.4940,5.6910,Bellingham,bellingham
1,usd,5.4390,5.7580,5.9610,6.1710,Bremerton,bremerton
2,usd,5.1950,5.4150,5.6270,6.2380,Olympia,olympia
3,usd,4.9390,5.2600,5.5090,5.8050,Richland-Kennewick-Pasco,richland-kennewick-pasco
4,usd,5.5320,5.8170,6.0480,6.3820,Seattle-Bellevue-Everett,seattle-bellevue-everett


In [16]:
    DATABASE_NAME = os.getenv('DATABASE_NAME')
    DATABASE_USER = os.getenv('DATABASE_USER')
    DATABASE_PORT = os.getenv('DATABASE_PORT')
    DATABASE_PASSWORD = os.getenv('DATABASE_PASSWORD')
    DATABASE_HOST = os.getenv('DATABASE_HOST')

    engine = create_engine(f'postgresql+psycopg2://{DATABASE_USER}:{DATABASE_PASSWORD}@{DATABASE_HOST}:{DATABASE_PORT}/{DATABASE_NAME}')

    with engine.connect() as conn:
        resort = conn.execute(text('select 1;'))
        for i in resort:
            print(i)

    cities.to_sql('cities',engine , if_exists='replace',index=False )

(1,)


14